In [14]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
import pandas as pd
import chromadb
import uuid

In [7]:
loader = WebBaseLoader("https://careers.nike.com/lead-technology-customer-support-engineer-analyst/job/R-80770")
page_data = loader.load().pop().page_content
print(page_data)





















Lead Technology Customer Support Engineer/Analyst












































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Re

In [ ]:
import os
os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key="GROQ_API_KEY",
    temperature=0,
)

In [6]:
prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ### INSTRUCTION:
    The scraped text is from the career's page of a website.
    Your job is to extract the job postings and return them in JSON format containing the 
    following keys: 'role', 'experience', 'skills' and 'description'.
    Only return the valid JSON.
    ### VALID JSON (NO PREAMBLE):
    """
)


chain_extract  = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})
print(res.content)

{
  "role": "Lead Technology Customer Support Engineer/Analyst",
  "experience": "3-5 years of experience with SAP, SAP BW, S/4 HANA, SAP AFS, Databricks, Snowflake, Oracle, Informatica, ServiceNow, Jira, and working knowledge of Autosys. Proven ability in data analysis, incident management, and security access governance.",
  "skills": [
    "Data analysis across SAP S/4 HANA and SAP AFS",
    "SAP, SAP BW, SAP AFS, S/4 HANA",
    "Databricks",
    "Snowflake",
    "Oracle",
    "Informatica",
    "ServiceNow",
    "Jira",
    "Autosys",
    "BI/BW and Batch Middleware frameworks",
    "SQL (Oracle)",
    "Active Directory entitlement management",
    "IDLocker and CUP security request processing",
    "SOX compliance auditing",
    "Incident and problem management (S1/S2)",
    "Release and upgrade coordination",
    "Leadership and team coordination across time zones",
    "Excellent organizational and communication skills",
    "Slack, email, calendar management"
  ],
  "descriptio

In [9]:
json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
type(json_res)
json_res

{'role': 'Lead Technology Customer Support Engineer/Analyst',
 'experience': '3-5 years of experience with SAP, SAP BW, S/4 HANA, SAP AFS, Databricks, Snowflake, Oracle, Informatica, ServiceNow, Jira, and working knowledge of Autosys. Proven ability in data analysis, incident management, and security access governance.',
 'skills': ['Data analysis across SAP S/4 HANA and SAP AFS',
  'SAP, SAP BW, SAP AFS, S/4 HANA',
  'Databricks',
  'Snowflake',
  'Oracle',
  'Informatica',
  'ServiceNow',
  'Jira',
  'Autosys',
  'BI/BW and Batch Middleware frameworks',
  'SQL (Oracle)',
  'Active Directory entitlement management',
  'IDLocker and CUP security request processing',
  'SOX compliance auditing',
  'Incident and problem management (S1/S2)',
  'Release and upgrade coordination',
  'Leadership and team coordination across time zones',
  'Excellent organizational and communication skills',
  'Slack, email, calendar management'],
 'description': 'The Lead Technology Customer Support Engineer

In [12]:
df = pd.read_csv('my_portfolio.csv')
df.head()

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio


In [15]:
client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name='portfolio')

if not collection.count():
    for _, row in df.iterrows():
        collection.add(
            documents=row["Techstack"],
            metadatas={"links":row["Links"]},
            ids=[str(uuid.uuid4())]
        )

In [20]:
job = json_res
links = collection.query(query_texts=job['skills'], n_results=1).get('metadatas')
links

[[{'links': 'https://example.com/ios-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'}],
 [{'links': 'https://example.com/flutter-portfolio'}],
 [{'links': 'https://example.com/android-tv-portfolio'}],
 [{'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/android-tv-portfolio'}],
 [{'links': 'https://example.com/kotlin-android-portfolio'}],
 [{'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'}],
 [{'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'}],
 [{'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'}],
 [{'links'

In [19]:
job = json_res
job['skills']

['Data analysis across SAP S/4 HANA and SAP AFS',
 'SAP, SAP BW, SAP AFS, S/4 HANA',
 'Databricks',
 'Snowflake',
 'Oracle',
 'Informatica',
 'ServiceNow',
 'Jira',
 'Autosys',
 'BI/BW and Batch Middleware frameworks',
 'SQL (Oracle)',
 'Active Directory entitlement management',
 'IDLocker and CUP security request processing',
 'SOX compliance auditing',
 'Incident and problem management (S1/S2)',
 'Release and upgrade coordination',
 'Leadership and team coordination across time zones',
 'Excellent organizational and communication skills',
 'Slack, email, calendar management']

In [ ]:
prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}

    ### INSTRUCTION:
    You are Mohit, a business development executive at MSP. MSP is an AI, Cloud, HPC Software & Security Consulting company dedicated to facilitating
    the seamless integration of business processes through automated tools. 
    We are New Gen Company focusing on Fast but reliable process optimization, cost reduction, and heightened overall efficiency. 
    Your job is to write a cold email to the client regarding the job mentioned above describing the capability of MSP 
    in fulfilling their needs.
    Also add the most relevant ones from the following links to showcase MSP's portfolio: {link_list}
    Remember you are Mohit, BDE at MSP. 
    Do not provide a preamble.
    ### EMAIL (NO PREAMBLE):

    """
)


chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print

**Subject:** Accelerate Nike’s SAP & Cloud Support Operations with MSP’s AI‑Driven Expertise  

Hi [Hiring Manager’s Name],

I’m Mohit, Business Development Executive at **MSP**, a next‑gen AI, Cloud, HPC, and Security consulting firm. We specialize in rapid, reliable process optimization that cuts costs while boosting operational efficiency—exactly the outcomes you’re seeking for the Lead Technology Customer Support Engineer/Analyst role.

### How MSP Aligns with Your Needs  

| Your Requirement | MSP Capability | Value Delivered |
|------------------|----------------|-----------------|
| **SAP S/4 HANA & AFS data‑flow analysis** | Deep‑dive analytics on SAP BW, S/4 HANA, and AFS using Python‑based ETL pipelines and AI‑driven anomaly detection. | Faster root‑cause identification, 30 % reduction in incident resolution time. |
| **Cloud data platforms (Databricks, Snowflake, Oracle)** | End‑to‑end migration & orchestration on Snowflake & Databricks, with automated schema sync and perfor